# Démo Complète: Pipeline Multi-Batch avec Nessie & Iceberg

**Scénario de démonstration d'entretien**

Cette démo simule un pipeline de données en production qui reçoit des fichiers tout au long de la journée.

## Plan de la démo

| Étape | Action | Ce qu'on montre |
|-------|--------|----------------|
| 1 | **Batch 1** (matin) | Ingestion initiale, création des tables |
| 2 | **Batch 2** (midi) | Append, mise à jour incrémentale |
| 3 | **Batch 3** (après-midi) | Append, évolution des données |
| 4 | **Time Travel** | Requêter les données à différents moments |
| 5 | **Rollback** | Revenir à un état précédent |
| 6 | **Branching** | Développement isolé sur branche dev |

**Prérequis**:
- Nessie démarré: `docker run -p 19120:19120 ghcr.io/projectnessie/nessie:latest`
- Données uploadées sur S3: `python scripts/upload_to_s3.py`

---
## INITIALISATION

In [1]:
# Configuration du chemin vers le projet
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import des modules Spark et configuration
from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET, NESSIE_URI

# Création de la session Spark
spark = get_spark("demo-complete")

print("=" * 60)
print("DÉMO LAKEHOUSE - INITIALISATION")
print("=" * 60)

DÉMO LAKEHOUSE - INITIALISATION


In [2]:
# Configuration de la branche Nessie main
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Création des namespaces (layers du medallion architecture)
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

print("Branche active: main")
print("Namespaces Nessie: bronze, silver, gold")
print(f"Bucket S3: {AWS_BUCKET}")

Branche active: main
Namespaces Nessie: bronze, silver, gold
Bucket S3: iceberg-nessie-lakehouse-demo


---
## BATCH 1 - PREMIÈRE INGESTION (MATIN)

**Scénario**: C'est le matin, le premier fichier de ventes arrive.

Fichier: `sales_batch_1.csv` (3,364 enregistrements)

In [3]:
# Import des fonctions PySpark
from pyspark.sql.functions import (
    current_timestamp, current_date, lit, col,
    sum as spark_sum, round as spark_round, count
)

# Nettoyage: suppression des tables si elles existent (pour une démo propre)
spark.sql("DROP TABLE IF EXISTS nessie.bronze.sales")
spark.sql("DROP TABLE IF EXISTS nessie.silver.sales")
spark.sql("DROP TABLE IF EXISTS nessie.gold.sales_by_category_region")

print("Tables existantes supprimées")

Tables existantes supprimées


In [4]:
# Lecture du premier batch depuis S3
batch1_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_001.csv"

batch1_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch1_path)
)

batch1_count = batch1_raw.count()
print(f"BATCH 1 lu: {batch1_count:,} enregistrements")

BATCH 1 lu: 3,364 enregistrements


In [5]:
# Enrichissement et création de la table Bronze
# Ajout de métadonnées: ingestion_date, ingestion_ts, source_file, source_system, batch_id
batch1_bronze = (
    batch1_raw
    .withColumn("ingestion_date", current_date())        # Date d'ingestion
    .withColumn("ingestion_ts", current_timestamp())      # Timestamp d'ingestion
    .withColumn("source_file", lit("sales_batch_1.csv")) # Nom du fichier source
    .withColumn("source_system", lit("sales_system"))    # Système source
    .withColumn("batch_id", lit("batch_1"))              # Identifiant du batch
)

# Création de la table Bronze avec partitionnement sur ingestion_date
batch1_bronze.writeTo("nessie.bronze.sales").using("iceberg").partitionedBy(col("ingestion_date")).create()

print(f"BATCH 1 -> Table Bronze créée: {batch1_count:,} enregistrements")

BATCH 1 -> Table Bronze créée: 3,364 enregistrements


In [6]:
# Transformation vers la couche Silver
# - Renommage des colonnes en snake_case
# - Typage fort des colonnes numériques
# - Filtrage des valeurs nulles
# - Suppression des doublons sur (order_id, product_id)
batch1_silver = (
    batch1_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    .dropDuplicates(["order_id", "product_id"])
)

batch1_silver.writeTo("nessie.silver.sales").using("iceberg").create()

silver_count = batch1_silver.count()
print(f"BATCH 1 -> Table Silver créée: {silver_count:,} enregistrements")

BATCH 1 -> Table Silver créée: 3,260 enregistrements


In [7]:
# Création de la table Gold: agrégation par catégorie et région
# Cette table contient les KPIs agrégés pour le reporting
batch1_gold = (
    batch1_silver
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

batch1_gold.writeTo("nessie.gold.sales_by_category_region").using("iceberg").create()

gold_count = batch1_gold.count()
print(f"BATCH 1 -> Table Gold créée: {gold_count:,} agrégats")

BATCH 1 -> Table Gold créée: 12 agrégats


In [8]:
# Affichage des résultats après Batch 1
print("\n" + "=" * 60)
print("RÉSULTATS APRÈS BATCH 1")
print("=" * 60)
print(f"Bronze: {spark.sql('SELECT COUNT(*) FROM nessie.bronze.sales').first()[0]:,} enregistrements")
print(f"Silver: {spark.sql('SELECT COUNT(*) FROM nessie.silver.sales').first()[0]:,} enregistrements")
print(f"Gold:   {spark.sql('SELECT COUNT(*) FROM nessie.gold.sales_by_category_region').first()[0]:,} agrégats")

print("\nTop 5 ventes par catégorie/region:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)


RÉSULTATS APRÈS BATCH 1
Bronze: 3,364 enregistrements
Silver: 3,260 enregistrements
Gold:   12 agrégats

Top 5 ventes par catégorie/region:
+---------------+------+-----------+-----------+
|category       |region|total_sales|order_count|
+---------------+------+-----------+-----------+
|Furniture      |West  |84314.97   |234        |
|Furniture      |East  |76048.63   |192        |
|Office Supplies|West  |75146.26   |598        |
|Technology     |West  |71744.79   |191        |
|Technology     |East  |70360.68   |167        |
+---------------+------+-----------+-----------+



In [9]:
# Sauvegarde des snapshot IDs pour le Time Travel (section suivante)
# Chaque opération Iceberg crée un snapshot unique qui permet de revenir à cet état
batch1_snapshot_bronze = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch1_snapshot_silver = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.silver.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch1_snapshot_gold = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.gold.sales_by_category_region.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

print("Snapshot ID Bronze (Batch 1):", batch1_snapshot_bronze)
print("Snapshot ID Silver (Batch 1):", batch1_snapshot_silver)
print("Snapshot ID Gold   (Batch 1):", batch1_snapshot_gold)
print("(Sauvegardes pour le Time Travel)")

Snapshot ID Bronze (Batch 1): 7825907608952333838
Snapshot ID Silver (Batch 1): 2384767348813450477
Snapshot ID Gold   (Batch 1): 4241962563570688563
(Sauvegardes pour le Time Travel)


---
## BATCH 2 - DEUXIÈME INGESTION (MIDI)

**Scénario**: C'est le midi, un nouveau fichier de ventes arrive.

Fichier: `sales_batch_2.csv` (3,501 enregistrements)

**Point clé**: Les nouvelles données sont **ajoutées** aux existantes (mode APPEND).

In [10]:
# Lecture du deuxieme batch depuis S3
batch2_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_002.csv"

batch2_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch2_path)
)

batch2_count = batch2_raw.count()
print(f"BATCH 2 lu: {batch2_count:,} enregistrements")

BATCH 2 lu: 3,501 enregistrements


In [11]:
# Enrichissement et ajout à la table Bronze (mode APPEND)
batch2_bronze = (
    batch2_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_2.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_2"))
)

# Mode APPEND: les données sont ajoutées aux existantes
batch2_bronze.writeTo("nessie.bronze.sales").append()

new_bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"BATCH 2 -> Ajouté à la table Bronze (APPEND)")
print(f"Nouveau total Bronze: {new_bronze_count:,} (+{new_bronze_count - batch1_count})")

BATCH 2 -> Ajouté à la table Bronze (APPEND)
Nouveau total Bronze: 6,865 (+3501)


In [12]:
# Transformation et ajout à Silver
batch2_silver = (
    batch2_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    .dropDuplicates(["order_id", "product_id"])
)

batch2_silver.writeTo("nessie.silver.sales").append()

new_silver_count = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
print(f"BATCH 2 -> Ajouté à la table Silver (APPEND)")
print(f"Nouveau total Silver: {new_silver_count:,} (+{new_silver_count - silver_count})")

BATCH 2 -> Ajouté à la table Silver (APPEND)
Nouveau total Silver: 6,587 (+3327)


In [13]:
# Recalcul complet de la table Gold (table de remplacement)
# Pour les tables d'agrégation, on recalcule tout à partir de Silver
all_silver = spark.table("nessie.silver.sales")

new_gold = (
    all_silver
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

# createOrReplace: remplace le contenu de la table
new_gold.writeTo("nessie.gold.sales_by_category_region").using("iceberg").createOrReplace()

print(f"Gold recalculé avec toutes les données de Silver: {new_gold.count():,} agrégats")

Gold recalculé avec toutes les données de Silver: 12 agrégats


In [14]:
# Affichage des résultats après Batch 2
print("\n" + "=" * 60)
print("RÉSULTATS APRÈS BATCH 2")
print("=" * 60)
print(f"Bronze: {spark.sql('SELECT COUNT(*) FROM nessie.bronze.sales').first()[0]:,} enregistrements")
print(f"Silver: {spark.sql('SELECT COUNT(*) FROM nessie.silver.sales').first()[0]:,} enregistrements")
print(f"Gold:   {spark.sql('SELECT COUNT(*) FROM nessie.gold.sales_by_category_region').first()[0]:,} agrégats")

print("\nTop 5 ventes par catégorie/region:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)


RÉSULTATS APRÈS BATCH 2
Bronze: 6,865 enregistrements
Silver: 6,587 enregistrements
Gold:   12 agrégats

Top 5 ventes par catégorie/region:
+---------------+------+-----------+-----------+
|category       |region|total_sales|order_count|
+---------------+------+-----------+-----------+
|Furniture      |West  |180579.27  |475        |
|Technology     |East  |173185.88  |342        |
|Technology     |West  |154526.5   |382        |
|Furniture      |East  |145835.02  |397        |
|Office Supplies|East  |139495.68  |1157       |
+---------------+------+-----------+-----------+



In [15]:
# Sauvegarde des snapshot IDs du Batch 2
batch2_snapshot_bronze = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch2_snapshot_silver = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.silver.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

batch2_snapshot_gold = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.gold.sales_by_category_region.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

print("Snapshot ID Bronze (Batch 2):", batch2_snapshot_bronze)
print("Snapshot ID Silver (Batch 2):", batch2_snapshot_silver)
print("Snapshot ID Gold   (Batch 2):", batch2_snapshot_gold)

Snapshot ID Bronze (Batch 2): 4977135241111746490
Snapshot ID Silver (Batch 2): 67499623872290859
Snapshot ID Gold   (Batch 2): 3900143024163522964


---
## BATCH 3 - TROISIÈME INGESTION (APRÈS-MIDI)

**Scénario**: C'est l'après-midi, un dernier fichier de ventes arrive.

Fichier: `sales_batch_3.csv` (3,500 enregistrements)

In [16]:
# Lecture du troisième batch depuis S3
batch3_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_003.csv"

batch3_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch3_path)
)

batch3_count = batch3_raw.count()
print(f"BATCH 3 lu: {batch3_count:,} enregistrements")

BATCH 3 lu: 3,500 enregistrements


In [17]:
# Enrichissement et ajout à Bronze
batch3_bronze = (
    batch3_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_3.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_3"))
)

batch3_bronze.writeTo("nessie.bronze.sales").append()

final_bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"BATCH 3 -> Ajouté à la table Bronze (APPEND)")
print(f"Nouveau total Bronze: {final_bronze_count:,} (+{final_bronze_count - batch1_count})")

BATCH 3 -> Ajouté à la table Bronze (APPEND)
Nouveau total Bronze: 10,365 (+7001)


In [18]:
# Transformation et ajout à Silver
batch3_silver = (
    batch3_bronze
    .select(
        col("Order ID").alias("order_id"),
        col("Order Date").alias("order_date"),
        col("Ship Date").alias("ship_date"),
        col("Ship Mode").alias("ship_mode"),
        col("Customer ID").alias("customer_id"),
        col("Customer Name").alias("customer_name"),
        col("Segment").alias("segment"),
        col("Country").alias("country"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Postal Code").alias("postal_code"),
        col("Region").alias("region"),
        col("Product ID").alias("product_id"),
        col("Category").alias("category"),
        col("Sub-Category").alias("sub_category"),
        col("Product Name").alias("product_name"),
        col("Sales").cast("double").alias("sales"),
        col("Quantity").cast("int").alias("quantity"),
        col("Discount").cast("double").alias("discount"),
        col("Profit").cast("double").alias("profit"),
        col("ingestion_date"),
        col("ingestion_ts"),
        col("batch_id")
    )
    .filter(col("order_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("sales").isNotNull())
    .dropDuplicates(["order_id", "product_id"])
)

batch3_silver.writeTo("nessie.silver.sales").append()

final_silver_count = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
print(f"BATCH 3 -> Ajouté à la table Silver (APPEND)")
print(f"Nouveau total Silver: {final_silver_count:,} (+{final_silver_count - batch1_count})")

BATCH 3 -> Ajouté à la table Silver (APPEND)
Nouveau total Silver: 9,917 (+6553)


In [19]:
# Recalcul de Gold avec toutes les données
all_silver_final = spark.table("nessie.silver.sales")

final_gold = (
    all_silver_final
    .groupBy("category", "region")
    .agg(
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(spark_sum("profit"), 2).alias("total_profit"),
        spark_sum("quantity").alias("total_quantity"),
        count("order_id").alias("order_count")
    )
)

final_gold.writeTo("nessie.gold.sales_by_category_region").using("iceberg").createOrReplace()

print(f"Gold recalculé final: {final_gold.count():,} agrégats")

Gold recalculé final: 12 agrégats


In [20]:
# Affichage des résultats finaux
print("\n" + "=" * 60)
print("RÉSULTATS APRÈS BATCH 3 (FINAL)")
print("=" * 60)
print(f"Bronze: {final_bronze_count:,} enregistrements")
print(f"Silver: {final_silver_count:,} enregistrements")
print(f"Gold:   {final_gold.count():,} agrégats")

print("\nTop 5 ventes par catégorie/region:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)


RÉSULTATS APRÈS BATCH 3 (FINAL)
Bronze: 10,365 enregistrements
Silver: 9,917 enregistrements
Gold:   12 agrégats

Top 5 ventes par catégorie/region:
+---------------+------+-----------+-----------+
|category       |region|total_sales|order_count|
+---------------+------+-----------+-----------+
|Technology     |East  |260324.31  |528        |
|Furniture      |West  |252284.98  |706        |
|Technology     |West  |250685.68  |595        |
|Office Supplies|West  |220154.6   |1883       |
|Furniture      |East  |206859.67  |595        |
+---------------+------+-----------+-----------+



---
## TIME TRAVEL - VOYAGE DANS LE TEMPS

**Point clé**: Avec Iceberg, on peut requêter les données telles qu'elles étaient à n'importe quel moment passé.

In [21]:
# Affichage de l'historique complet des snapshots Bronze
print("=" * 60)
print("HISTORIQUE DES SNAPSHOTS BRONZE")
print("=" * 60 + "\n")

spark.sql("""
    SELECT 
        made_current_at,
        snapshot_id,
        parent_id
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""").show(truncate=False)

HISTORIQUE DES SNAPSHOTS BRONZE

+-----------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |
+-----------------------+-------------------+-------------------+
|2026-03-17 00:33:19.846|7825907608952333838|NULL               |
|2026-03-17 00:33:29.54 |4977135241111746490|7825907608952333838|
|2026-03-17 00:33:38.136|262581489176584360 |4977135241111746490|
+-----------------------+-------------------+-------------------+



In [22]:
# Time Travel 1: Requêter les données du Batch 1 uniquement
print("\n" + "=" * 60)
print("TIME TRAVEL: DONNÉES APRÈS BATCH 1")
print("=" * 60 + "\n")

# Utilisation de VERSION AS OF pour requêter un état passé
count_after_batch1 = spark.sql(f"""
    SELECT COUNT(*) as cnt 
    FROM nessie.bronze.sales VERSION AS OF '{batch1_snapshot_bronze}'
""").first()[0]

print(f"Nombre d'enregistrements après Batch 1: {count_after_batch1:,}")

spark.sql(f"""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales VERSION AS OF '{batch1_snapshot_bronze}'
    GROUP BY batch_id
    ORDER BY batch_id
""").show()


TIME TRAVEL: DONNÉES APRÈS BATCH 1

Nombre d'enregistrements après Batch 1: 3,364
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
+--------+-----+



In [23]:
# Time Travel 2: Requêter les données du Batch 2
print("\n" + "=" * 60)
print("TIME TRAVEL: DONNÉES APRÈS BATCH 2")
print("=" * 60 + "\n")

count_after_batch2 = spark.sql(f"""
    SELECT COUNT(*) as cnt 
    FROM nessie.bronze.sales VERSION AS OF '{batch2_snapshot_bronze}'
""").first()[0]

print(f"Nombre d'enregistrements après Batch 2: {count_after_batch2:,}")

spark.sql(f"""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales VERSION AS OF '{batch2_snapshot_bronze}'
    GROUP BY batch_id
    ORDER BY batch_id
""").show()


TIME TRAVEL: DONNÉES APRÈS BATCH 2

Nombre d'enregistrements après Batch 2: 6,865
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
+--------+-----+



In [24]:
# Time Travel 3: Données actuelles (après Batch 3)
print("\n" + "=" * 60)
print("DONNÉES ACTUELLES (APRÈS BATCH 3)")
print("=" * 60 + "\n")

count_after_batch3 = spark.sql("""
    SELECT COUNT(*) as cnt
    FROM nessie.bronze.sales
""").first()[0]

print(f"Nombre d'enregistrements après Batch 3: {count_after_batch3:,}")

spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("Observation: Chaque batch crée un nouveau snapshot !")
print("On peut revenir à n'importe quel moment dans le temps.")


DONNÉES ACTUELLES (APRÈS BATCH 3)

Nombre d'enregistrements après Batch 3: 10,365
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
| batch_3| 3500|
+--------+-----+

Observation: Chaque batch crée un nouveau snapshot !
On peut revenir à n'importe quel moment dans le temps.


In [25]:
# Comparaison des agrégats Gold dans le temps
print("\n" + "=" * 60)
print("ÉVOLUTION DES VENTES TECHNOLOGY/EAST")
print("=" * 60 + "\n")

# Ventes Technology/East après Batch 1 (utilisant le snapshot GOLD)
sales_batch1 = spark.sql(f"""
    SELECT total_sales 
    FROM nessie.gold.sales_by_category_region VERSION AS OF '{batch1_snapshot_gold}'
    WHERE category = 'Technology' AND region = 'East'
""").first()[0]

# Ventes Technology/East actuelles
sales_current = spark.sql("""
    SELECT total_sales 
    FROM nessie.gold.sales_by_category_region
    WHERE category = 'Technology' AND region = 'East'
""").first()[0]

print(f"Ventes Technology/East après Batch 1: {sales_batch1:,.2f}")
print(f"Ventes Technology/East actuelles:     {sales_current:,.2f}")
print(f"Augmentation:                         {sales_current - sales_batch1:,.2f}")


ÉVOLUTION DES VENTES TECHNOLOGY/EAST

Ventes Technology/East après Batch 1: 70,360.68
Ventes Technology/East actuelles:     260,324.31
Augmentation:                         189,963.63


---
## ROLLBACK - RETOUR À UN ÉTAT PRÉCÉDENT

**Point clé**: Avec Iceberg, on peut revenir à un état précédent en cas d'erreur ou de problème.

In [26]:
# État actuel avant rollback (après Batch 3)
print("=" * 60)
print("ÉTAT ACTUEL (APRÈS BATCH 3)")
print("=" * 60 + "\n")

bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
silver_count = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
gold_count = spark.sql("SELECT COUNT(*) FROM nessie.gold.sales_by_category_region").first()[0]

print(f"Bronze: {bronze_count:,} enregistrements")
print(f"Silver: {silver_count:,} enregistrements")
print(f"Gold:   {gold_count:,} agrégats")

current_snapshot = spark.sql("""
    SELECT snapshot_id 
    FROM nessie.bronze.sales.snapshots 
    ORDER BY committed_at DESC 
    LIMIT 1
""").first()[0]

print(f"Snapshot actuel: {current_snapshot}")

ÉTAT ACTUEL (APRÈS BATCH 3)

Bronze: 10,365 enregistrements
Silver: 9,917 enregistrements
Gold:   12 agrégats
Snapshot actuel: 262581489176584360


In [27]:
# ROLLBACK vers l'état après Batch 2
print("\n" + "=" * 60)
print("ROLLBACK VERS L'ÉTAT APRÈS BATCH 2")
print("=" * 60 + "\n")

print(f"Snapshot cible: {batch2_snapshot_bronze}")

# Lecture des données à l'état du Batch 2
bronze_batch2 = spark.sql(f"""
    SELECT * FROM nessie.bronze.sales VERSION AS OF '{batch2_snapshot_bronze}'
""")

count_batch2 = bronze_batch2.count()
print(f"Données à restaurer: {count_batch2:,} enregistrements")

# Rollback: remplacement du contenu de la table par l'état du Batch 2
bronze_batch2.writeTo("nessie.bronze.sales").using("iceberg").createOrReplace()

print("\nRollback Bronze terminé!")


ROLLBACK VERS L'ÉTAT APRÈS BATCH 2

Snapshot cible: 4977135241111746490
Données à restaurer: 6,865 enregistrements

Rollback Bronze terminé!


In [28]:
# Vérification de l'état après rollback Bronze
print("\n" + "=" * 60)
print("ÉTAT APRÈS ROLLBACK BRONZE")
print("=" * 60 + "\n")

bronze_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Bronze: {bronze_after_rollback:,} enregistrements")
print(f"(Avant rollback: {bronze_count:,})")
print(f"Différence: {bronze_count - bronze_after_rollback:,} enregistres supprimés")

print("\nBatches présents après rollback:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()


ÉTAT APRÈS ROLLBACK BRONZE

Bronze: 6,865 enregistrements
(Avant rollback: 10,365)
Différence: 3,500 enregistres supprimés

Batches présents après rollback:
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
+--------+-----+



In [29]:
# ROLLBACK complet: tables Silver et Gold
print("\n" + "=" * 60)
print("ROLLBACK DES TABLES SILVER ET GOLD")
print("=" * 60 + "\n")

# Rollback Silver
silver_batch2 = spark.sql(f"""
    SELECT * FROM nessie.silver.sales VERSION AS OF '{batch2_snapshot_silver}'
""")
silver_batch2.writeTo("nessie.silver.sales").using("iceberg").createOrReplace()
print("Rollback Silver terminé!")

# Rollback Gold
gold_batch2 = spark.sql(f"""
    SELECT * FROM nessie.gold.sales_by_category_region VERSION AS OF '{batch2_snapshot_gold}'
""")
gold_batch2.writeTo("nessie.gold.sales_by_category_region").using("iceberg").createOrReplace()

print("Rollback Gold terminé!")


ROLLBACK DES TABLES SILVER ET GOLD

Rollback Silver terminé!
Rollback Gold terminé!


In [30]:
# Vérification de l'état après rollback Silver et Gold
print("\n" + "=" * 60)
print("ÉTAT APRÈS ROLLBACK SILVER ET GOLD")
print("=" * 60 + "\n")

silver_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.silver.sales").first()[0]
gold_after_rollback = spark.sql("SELECT COUNT(*) FROM nessie.gold.sales_by_category_region").first()[0]

print(f"Silver: {silver_after_rollback:,} enregistrements")
print(f"(Avant rollback: {silver_count:,})")
print(f"Différence: {silver_count - silver_after_rollback:,} enregistres supprimés\n")

print(f"Gold: {gold_after_rollback:,} agrégats")
print(f"(Avant rollback: {gold_count:,})")

print("\nBatches présents dans Silver après rollback:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.silver.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("Aperçu des données Gold après rollback:")
spark.sql("""
    SELECT category, region, total_sales, order_count
    FROM nessie.gold.sales_by_category_region
    ORDER BY total_sales DESC
    LIMIT 5
""").show(truncate=False)

print("\nTous les rollbacks sont terminés et vérifiés!")
print("Les données sont revenues à l'état après Batch 2.")


ÉTAT APRÈS ROLLBACK SILVER ET GOLD

Silver: 6,587 enregistrements
(Avant rollback: 9,917)
Différence: 3,330 enregistres supprimés

Gold: 12 agrégats
(Avant rollback: 12)

Batches présents dans Silver après rollback:
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3260|
| batch_2| 3327|
+--------+-----+

Aperçu des données Gold après rollback:
+---------------+------+-----------+-----------+
|category       |region|total_sales|order_count|
+---------------+------+-----------+-----------+
|Furniture      |West  |180579.27  |475        |
|Technology     |East  |173185.88  |342        |
|Technology     |West  |154526.5   |382        |
|Furniture      |East  |145835.02  |397        |
|Office Supplies|East  |139495.68  |1157       |
+---------------+------+-----------+-----------+


Tous les rollbacks sont terminés et vérifiés!
Les données sont revenues à l'état après Batch 2.


---
## BRANCHING NESSIE - VERSIONNING GIT-LIKE POUR LES DATA LAKES

### Concept

**Nessie** apporte le versionnage de type Git pour les tables du data lake. Comme Git pour le code, Nessie permet :

* de créer des **branches** isolées pour expérimenter
* de tester des transformations de données sans impacter la production
* de **fusionner (merge)** les modifications validées dans la branche principale

### Cas d'usage

Un data engineer veut tester une nouvelle transformation ou ingérer un nouveau batch de données **sans risquer** de cycler la production. Il crée une branche de travail, fait ses tests, puis merge si tout est OK.

```
┌─────────┐         ┌─────────┐
│  main   │ ←------ │   dev   │
│ (prod)  │  merge  │ (test)  │
└─────────┘         └─────────┘
```

#### 1. Lister les références (branches) existantes

**Commande**: `LIST REFERENCES IN nessie`

Cette commande affiche toutes les branches présentes dans le catalogue Nessie. Par défaut, seule la branche `main` existe.

In [31]:
# Nettoyage de la branche dev si elle existe (pour une démo propre)
spark.sql("DROP BRANCH IF EXISTS dev IN nessie")

print("=" * 60)
print("1. LISTE DES RÉFÉRENCES (BRANCHES) NESSIE")
print("=" * 60 + "\n")

spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

print("Explication:")
print("- Affiche toutes les branches du catalogue Nessie")
print("- Par défaut, seule 'main' existe (branche de production)")

1. LISTE DES RÉFÉRENCES (BRANCHES) NESSIE

+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|e19320eb75f8e2e5ecf445a1de125fade0f99a69b5cbab17bcbe223416e5b867|
+-------+----+----------------------------------------------------------------+

Explication:
- Affiche toutes les branches du catalogue Nessie
- Par défaut, seule 'main' existe (branche de production)


#### 2. Voir la branche active

**Commande**: `SHOW REFERENCE IN nessie`

Cette commande affiche la branche sur laquelle nous sommes actuellement positionnés. Toutes les opérations SQL seront effectuées sur cette branche.

In [32]:
# Retour sur la branche main
spark.sql("USE REFERENCE main IN nessie")

print("\n" + "=" * 60)
print("2. BRANCHE ACTIVE")
print("=" * 60 + "\n")

spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

current_ref = spark.sql("SHOW REFERENCE IN nessie").first()[1]
print(f"Nous sommes sur la branche: {current_ref}")
print("Toutes les opérations SQL affecteront cette branche tant qu'on ne bascule pas sur une autre branche.")


2. BRANCHE ACTIVE

+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|e19320eb75f8e2e5ecf445a1de125fade0f99a69b5cbab17bcbe223416e5b867|
+-------+----+----------------------------------------------------------------+

Nous sommes sur la branche: main
Toutes les opérations SQL affecteront cette branche tant qu'on ne bascule pas sur une autre branche.


#### 3. Créer une branche de travail

**Commande**: `CREATE BRANCH dev IN nessie FROM main`

Crée une nouvelle branche nommée `dev` à partir de `main`. Cette branche est initialement identique à `main` (copie instantanée, pas de copie de données).

In [33]:
print("\n" + "=" * 60)
print("3. CRÉATION DE LA BRANCHE 'dev'")
print("=" * 60 + "\n")

try:
    spark.sql("CREATE BRANCH dev IN nessie FROM main").show(truncate=False)
    print("Branche 'dev' créée avec succès!")
except Exception as e:
    if "already exists" in str(e).lower():
        print("\nLa branche 'dev' existe déjà (créée précédemment)")
    else:
        print(f"\nInfo: {e}")

print("\nExplication:")
print("- La branche 'dev' est une copie logique de 'main'")
print("- Pas de copie physique des données (instantané)")
print("- Les modifications sur 'dev' n'impacteront pas 'main'")


3. CRÉATION DE LA BRANCHE 'dev'

+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |dev |e19320eb75f8e2e5ecf445a1de125fade0f99a69b5cbab17bcbe223416e5b867|
+-------+----+----------------------------------------------------------------+

Branche 'dev' créée avec succès!

Explication:
- La branche 'dev' est une copie logique de 'main'
- Pas de copie physique des données (instantané)
- Les modifications sur 'dev' n'impacteront pas 'main'


#### 4. Basculer sur la branche dev

**Commande**: `USE REFERENCE dev IN nessie`

Change la branche active pour `dev`. Maintenant, toutes les opérations (INSERT, UPDATE, CREATE TABLE) seront effectuées sur cette branche, pas sur `main`.

In [34]:
print("\n" + "=" * 60)
print("4. BASCULE VERS LA BRANCHE 'dev'")
print("=" * 60 + "\n")

spark.sql("USE REFERENCE dev IN nessie")

# Vérification que nous sommes bien sur dev
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

print("Explication:")
print("- Nous sommes maintenant sur la branche 'dev'")
print("- Toutes les prochaines opérations affecteront 'dev'")
print("- La branche 'main' reste intacte (production safe)")


4. BASCULE VERS LA BRANCHE 'dev'

+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |dev |e19320eb75f8e2e5ecf445a1de125fade0f99a69b5cbab17bcbe223416e5b867|
+-------+----+----------------------------------------------------------------+

Explication:
- Nous sommes maintenant sur la branche 'dev'
- Toutes les prochaines opérations affecteront 'dev'
- La branche 'main' reste intacte (production safe)


#### 5. Faire une modification sur la branche dev

Maintenant que nous sommes sur `dev`, ajoutons le Batch 3 pour tester notre pipeline. Cette modification n'impactera pas `main`.

In [35]:
print("\n" + "=" * 60)
print("5. MODIFICATION SUR LA BRANCHE 'dev' (AJOUT DU BATCH 3)")
print("=" * 60 + "\n")

# Vérification de l'état actuel sur dev
count_dev_before = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Données Bronze sur 'dev' avant: {count_dev_before:,} enregistrements")

# Lecture et ajout du Batch 3 sur la branche dev
batch3_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_003.csv"

batch3_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch3_path)
)

batch3_bronze = (
    batch3_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_3.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_3"))
)

batch3_bronze.writeTo("nessie.bronze.sales").append()

count_dev_after = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Données Bronze sur 'dev' après:  {count_dev_after:,} enregistrements")
print(f"Différence: +{count_dev_after - count_dev_before:,} enregistrements")

print("\nCette modification est uniquement sur 'dev' - 'main' n'est pas impactée!")


5. MODIFICATION SUR LA BRANCHE 'dev' (AJOUT DU BATCH 3)

Données Bronze sur 'dev' avant: 6,865 enregistrements
Données Bronze sur 'dev' après:  10,365 enregistrements
Différence: +3,500 enregistrements

Cette modification est uniquement sur 'dev' - 'main' n'est pas impactée!


#### 6. Comparer main vs dev

Vérifions que `main` n'a pas été modifiée en comparant les données des deux branches.

In [ ]:
print("\n" + "=" * 60)
print("6. COMPARAISON: main vs dev")
print("=" * 60 + "\n")

# Données sur dev (où nous sommes)
dev_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]

# Bascule vers main pour comparer
spark.sql("USE REFERENCE main IN nessie")
main_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]

print(f"BRANCHE 'dev' (avec Batch 3): {dev_count:,} enregistrements")
print(f"BRANCHE 'main' (production):  {main_count:,} enregistrements")
print(f"\nDifférence: {dev_count - main_count:,} enregistrements de plus sur 'dev'")

# Affichage des batches présents sur main
print("\nBatches présents sur 'main':")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("La branche 'main' est intacte! Les modifications sur 'dev' n'ont pas impacté la production.")


6. COMPARAISON: main vs dev

BRANCHE 'dev' (avec Batch 3): 10,365 enregistrements
BRANCHE 'main' (production):  6,865 enregistrements

Différence: 3,500 enregistrements de plus sur 'dev'

Batches présents sur 'main':
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
+--------+-----+

La branche 'main' est intacte! Les modifications sur 'dev'
n'ont pas impacté la production.


#### 7. Afficher l'historique des commits

**Commande**: `SHOW LOG IN nessie`

Affiche l'historique des commits sur la branche active, similaire à `git log`. On voit toutes les opérations effectuées avec leurs métadonnées.

In [37]:
print("\n" + "=" * 60)
print("7. HISTORIQUE DES COMMITS SUR 'main'")
print("=" * 60 + "\n")

spark.sql("SHOW LOG IN nessie").show(truncate=False)

print("Explication:")
print("- Chaque opération (CREATE, INSERT, MERGE) crée un commit")
print("- Les commits ont un ID, un auteur, un timestamp et un message")
print("- Comme Git, on peut naviguer dans l'historique !")


7. HISTORIQUE DES COMMITS SUR 'main'

+------+---------+----------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+--------------------------+--------------------------+-----------------------------------------------------------------------------------------+
|author|committer|hash                                                            |message                                                                                                                                                     |signedOffBy|authorTime                |committerTime             |properties                                                                               |
+------+---------+----------------------------------------------------------------+------------------------------------------------------------------------------------

#### 8. Fusionner la branche dev dans main

**Commande**: `MERGE BRANCH dev INTO main IN nessie`

Une fois les tests validés sur `dev`, on fusionne la branche dans `main`. Cela applique toutes les modifications de `dev` à la production.

In [38]:
print("\n" + "=" * 60)
print("8. FUSION DE 'dev' DANS 'main'")
print("=" * 60 + "\n")

print("On fusionne la branche dans 'main' pour déployer en production.")

spark.sql("MERGE BRANCH dev INTO main IN nessie").show(truncate=False)

# Important: rafraîchir les métadonnées après un merge Nessie
spark.sql("REFRESH TABLE nessie.bronze.sales")

# Vérification
main_count_after = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"\n'main' après fusion: {main_count_after:,} enregistrements")
print(f"(avant fusion: {main_count:,}, +{main_count_after - main_count:,} nouveaux)")

print("\nBatches présents sur 'main' après fusion:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("Le Batch 3 est maintenant présent en production!")


8. FUSION DE 'dev' DANS 'main'

On fusionne la branche dans 'main' pour déployer en production.
+----+----------------------------------------------------------------+
|name|hash                                                            |
+----+----------------------------------------------------------------+
|main|6dbcbc3ac0422cbb6e672a98ea754123dcd51a976a50464a4b11c97877ba8fbe|
+----+----------------------------------------------------------------+


'main' après fusion: 10,365 enregistrements
(avant fusion: 6,865, +3,500 nouveaux)

Batches présents sur 'main' après fusion:
+--------+-----+
|batch_id|count|
+--------+-----+
| batch_1| 3364|
| batch_2| 3501|
| batch_3| 3500|
+--------+-----+

Le Batch 3 est maintenant présent en production!


---
## RÉSUMÉ DE LA DÉMO

### Fonctionnalités démontrées

| Fonctionnalité | Description |
|---------------|-------------|
| **Ingestion multi-batch** | Ajout progressif de données avec mode APPEND |
| **Time Travel** | Requêter les données à n'importe quel moment passé via snapshots |
| **Rollback** | Revenir à un état précédent (ex: annuler un batch) |
| **Branching** | Créer des branches isolées pour expérimenter sans risque |
| **Merge** | Fusionner une branche expérimentale dans main |

### Avantages Nessie + Iceberg + AWS

* **Isolation**: Travailler sur une branche dev sans impacter la production
* **Sécurité**: Annulation possible des erreurs via rollback
* **Audit**: Historique complet de toutes les modifications
* **Flexibilité**: Tester des transformations avant de les déployer

**Fin de la démo !**